# back-fn-call-with-recipe-args — worked example 1: Call the right back_fn for one parent via (argnum) dispatch

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `back-fn-call-with-recipe-args`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The canonical backward call is `back_fn(grad_out, node.array, *recipe.args, **recipe.kwargs)`. In a real autograd, one forward op (e.g. `multiply`) registers a *different* back_fn per `argnum` (one for `x`, one for `y`). Dispatch looks up the back_fn for `(forward_fn, argnum)`, then invokes it with exactly that four-channel shape so the cached `out` and the original raw args/kwargs reach the reverse pass.

## Worked solution

**Goal.** Given a node produced by `multiply(x, y)`, compute the gradient flowing into parent `argnum=0` (i.e. `x`).

**Step 1 — register the back_fns.** `multiply_back0(grad_out, out, x, y)` returns `dL/dx = grad_out * y`; `multiply_back1` returns `grad_out * x`. We store them in a dict keyed by `(forward_fn, argnum)`. This mirrors the real `BACK_FUNCS` registry.

**Step 2 — look up the back_fn.** `back_fn = BACK_FUNCS[(node.recipe.func, argnum)]`. The recipe remembers which forward function created this node, so dispatch is just a dict read.

**Step 3 — invoke with the canonical shape.** `back_fn(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)`. Note `node.array` is the *raw tensor* `out`, not the MiniTensor. The `*` splat unpacks `(x, y)` into two positionals; `**` unpacks the (empty here) kwargs. `multiply_back0` reads `y` (the second positional after `out`) and returns `grad_out * y`.

**Why it works.** The chain rule for `x*y` w.r.t. `x` is `y`, so `dL/dx = grad_out * y`. Because we passed the *original* `x` and `y` through `*recipe.args`, the back_fn has everything it needs without recomputing the forward.

In [ ]:
BACK_FUNCS = {}

def register(forward_fn, argnum, back_fn):
    BACK_FUNCS[(forward_fn, argnum)] = back_fn

def multiply(x, y):
    return x * y

def multiply_back0(grad_out, out, x, y):
    return grad_out * y

def multiply_back1(grad_out, out, x, y):
    return grad_out * x

register(multiply, 0, multiply_back0)
register(multiply, 1, multiply_back1)

class Recipe:
    def __init__(self, func, args, kwargs):
        self.func, self.args, self.kwargs = func, args, kwargs

class Node:
    def __init__(self, array, recipe):
        self.array, self.recipe = array, recipe

def backward_into_parent(node, grad_out, argnum):
    back_fn = BACK_FUNCS[(node.recipe.func, argnum)]
    return back_fn(
        grad_out,
        node.array,
        *node.recipe.args,
        **node.recipe.kwargs,
    )

t.manual_seed(0)
x = t.randn(2, 3)
y = t.randn(2, 3)
out = multiply(x, y)
node = Node(out, Recipe(multiply, (x, y), {}))
grad_out = t.ones_like(out)

dx = backward_into_parent(node, grad_out, 0)
dy = backward_into_parent(node, grad_out, 1)
print('dx matches y:', t.allclose(dx, y))
print('dy matches x:', t.allclose(dy, x))